per neuron alignment

In [4]:
import torch
import pickle
import numpy
import re
from itertools import combinations
import pandas
import os
from collections import Counter
def  get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts
import pickle
with open("/workspace/CCE_NLI/code/Abstractions/final_abstractions.pkl", 'rb') as f:
    abs_map = pickle.load(f)

def find_cluster(raw_concept):
    for cluster in abs_map:
        if raw_concept in abs_map[cluster]:
            return cluster
def find_abstractions(expls):
    if isinstance(expls, set):
        glob = list(expls)
    else:
        glob=expls
    abstracts=[]
    exact_concepts_perabs=defaultdict(list)
    for concept in glob:
        raw_concept = concept.split(":")[-1]
        abstraction_cluster = find_cluster(raw_concept)
        if abstraction_cluster==143: 
            continue
        if not abstraction_cluster:abstraction_cluster=150
        exact_concepts_perabs[abstraction_cluster].append(raw_concept)
        
        abstracts.append(abstraction_cluster)
    for a in exact_concepts_perabs:
        exact_concepts_perabs[a] = Counter(exact_concepts_perabs[a])
    return Counter(abstracts), set(abstracts), dict(
                                                    sorted(
                                                        exact_concepts_perabs.items(),
                                                        key=lambda x: sum(x[1].values()),
                                                        reverse=True
                                                    )
)


def get_groups(formula):
    indiv = get_indiv_concepts(formula)
    groups = []
    for length in range(2, 3):
        for combo in combinations(indiv, length):
            groups.append(tuple(sorted(combo)))
    return sorted(groups)

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts  = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(lambda: defaultdict(set))
    s=set()
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[csv_file.split("/")[-2]][csv_file.split("/")[-1]].update(get_indiv_concepts(formula))
            s.update(get_indiv_concepts(formula))
     
      
    return concept_dict, s


In [19]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def all_correct_to_wrong(dense_cw, sparse_cw):
  
    
    return set(dense_cw['correct']) & set(sparse_cw['wrong'])
        

def find_highest_activating_neuron(samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
    flweights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    flweights=flweights.reshape((3,flweights.shape[0]//3 ))
    contribution = torch.tensor(samples_activations).abs().squeeze(0) *flweights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    most_impactful = contribution[total].argmax().item()
    
    return [most_impactful]
    

def find_highest_activating_neuron_at_cluster(cluster_mask, samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
   
    
    fl_cluster_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    
    fl_cluster_weights=fl_cluster_weights.reshape((3,fl_cluster_weights.shape[0]//3 ))
  
    contribution = torch.tensor(samples_activations) * torch.tensor(cluster_mask) *fl_cluster_weights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    most_impactful = contribution[total].argmax().item()
    
    return [most_impactful]
    
def find_highest_activating_neurons(samples_activations, model_finallayerweights, d):
    flweights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    flweights = flweights.reshape((3, flweights.shape[0]//3))
    
    contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights
    
    total = contribution.sum(dim=1).argmax()  # best class row
    
    class_contributions = contribution[total]  # [1024]
    
    sorted_neurons = class_contributions.argsort(descending=True)  # indices sorted by contribution
    active_sorted_neurons = sorted_neurons[class_contributions[sorted_neurons] > 0]  # filter only active
    
   
    return active_sorted_neurons.tolist()
    
def find_highest_activating_neurons_at_cluster(cluster_mask, samples_activations,
                                                model_finallayerweights, dead_neurons=None):
    #fl_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    #fl_weights = fl_weights.reshape((3, fl_weights.shape[0]//3))
    
    # shape (3, 1024) — keep full matrix, zeros contribute nothing naturally
    fl_weights = model_finallayerweights.detach().cpu()  # (3, 1024)

    activations = torch.tensor(samples_activations).abs()  # (1024,)
    mask        = torch.tensor(cluster_mask)                # (1024,)

    
    # contribution per class per neuron: (3, 1024)
    contribution = activations.unsqueeze(0) * mask.unsqueeze(0) * fl_weights
    #print(torch.whe)
    # #only neurons that are active at this cluster

    # optionally zero out known dead neurons

    # pick the class with highest total contribution
    winning_class    = contribution.sum(dim=1).argmax()   # scalar
    class_contribs   = contribution[winning_class]        # (1024,)
    
    
    # rank neurons by contribution, keep only those with positive contribution
    sorted_neurons   = class_contribs.argsort(descending=True)
    
    active_neurons   = sorted_neurons[class_contribs[sorted_neurons] > 0]
    
    #print(1024 - torch.where(class_contribs[sorted_neurons] != 0,1,0).sum())
    
    return [a for a in active_neurons.tolist() if a not in dead_neurons]

def find_activating_neurons_at_cluster(cluster_mask, samples_activations,
                                                 dead_neurons=None):
    #fl_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    #fl_weights = fl_weights.reshape((3, fl_weights.shape[0]//3))
    
    # shape (3, 1024) — keep full matrix, zeros contribute nothing naturally
   

    activations = torch.tensor(samples_activations).abs()  # (1024,)
    mask        = torch.tensor(cluster_mask)                # (1024,)

    
    # contribution per class per neuron: (3, 1024)
    contribution = activations.unsqueeze(0) * mask.unsqueeze(0)
    contribution= contribution.squeeze(0)
    contribution = torch.where(contribution>0,1,0)
    
    
    #print(1024 - torch.where(class_contribs[sorted_neurons] != 0,1,0).sum())
    
    return [i for i,a in enumerate(contribution) if a > 0 and a not in dead_neurons]

def activationsiou(a,b):
    a = torch.where(a>0, 1,0)
    b = torch.where(b>0, 1,0)
    return (a&b).sum() / (a|b).sum() 
    
def iou(a,b):
    if isinstance(a, torch.Tensor):
        return (a&b).sum() / (a|b).sum()
    return len(a&b) / len(a|b) if len(a|b) > 0 else 0
ignore=0
def percent_sparse_in_dense(s,d):
    if len(d)==0 or len(s)==0: 
        return 1
    if len(d)==0 or len(s)==0: return 0
    return len(s&d)/len(d)

def concept_diff(expl_dense, expl_sparse):
    overlap = iou(set(expl_dense), set(expl_sparse))
    return 1-overlap

def get_mask(root,  cluster):
    return torch.load(os.path.join(root, f'Cluster{cluster}masks.pt'), map_location=device).t()

def get_subactiv(mask, cluster):
    dead=[]
    for n,i in enumerate(mask):
        if i.sum() < 500:
            dead.append(n)
    return dead

In [ ]:
localtion

In [25]:
import os
import pickle
import torch
from collections import defaultdict
import numpy as np

model  = 'BERT'
method = 'wanda'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── discover all Run* directories ─────────────────────────────────────────────
exp_root = f"/workspace/CCE_NLI/{model}/exp/{method}/"
run_dirs = sorted([
    d for d in os.listdir(exp_root)
    if os.path.isdir(os.path.join(exp_root, d))
    and d.startswith('Run')
    and '4cl' not in d
])
print(f"Found runs: {run_dirs}")

# ── collect sparsity levels PER RUN (so index alignment is run-local) ─────────
run_sparsity_levels = {}
for run in run_dirs:
    expls_dir = os.path.join(exp_root, run, 'Expls')
    levels = []
    if os.path.exists(expls_dir):
        for s in sorted(os.listdir(expls_dir)):
            if '.ipynb' not in s and '0.0%Pruned' not in s:
                levels.append(s)
    run_sparsity_levels[run] = levels
    print(f"  {run}: {levels}")

max_iters    = max(len(v) for v in run_sparsity_levels.values())
iter_indices = list(range(1, max_iters + 1))
print(f"Max iterations: {max_iters}")

# ── accumulators: iter_index -> metric lists across runs ──────────────────────
results_across_runs = {
    idx: {
        'cluster_holistic':     defaultdict(list),
        'cluster_per_neuron':   defaultdict(list),
        'concepts_not_encoded': defaultdict(list),
        'percent_of_dense_firing_concepts_preserved_in_sparse': defaultdict(list),
        'concept_hol_all':      [],
        'per_neuron_all':       [],
        'sparsity_labels':      [],   # one entry per run for display
    }
    for idx in iter_indices
}

dense_run_dir  = 'Run0.25_5'
dense_run_path = os.path.join(f"/workspace/CCE_NLI/{model}/exp/lottery_ticket", dense_run_dir)

# ── dense activations ─────────────────────────────────────────────────────────
dense_activ_path = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/{dense_run_dir}/0_Pruning_Iter/final_layer_activations.pkl"
if not os.path.exists(dense_activ_path):
    print(f"  Dense activations not found, skipping.")

with open(dense_activ_path, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls_root = os.path.join(dense_run_path, 'Expls', '0.0%Pruned')
dense_expls = {}
for cl in [1, 2, 3]:
    e, _ = load_csv_data(os.path.join(dense_expls_root, f'Cluster{cl}IOUS1024N.csv'))

    dense_expls[cl] = e

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_run_path, 'Masks', '0.0%Pruned')
dense_masks = {}
dense_dead={}
for cl in [1, 2, 3]:
    dense_masks[cl] = get_mask(dense_mask_path, cluster=cl).t()
    dense_dead[cl]=get_subactiv(dense_masks[cl], cl)


clusters_dense = {cl: (dense_masks[cl], dense_expls[cl]) for cl in [1, 2, 3]}

# ── main loop over runs ───────────────────────────────────────────────────────
for run in run_dirs:
    print(f"\n{'='*60}")
    print(f"RUN: {run}")
    print(f"{'='*60}")

    run_path = os.path.join(exp_root, run)

    # iterate over THIS RUN's own sorted sparsity levels — index is the key
    for start, sparsity in enumerate(run_sparsity_levels[run], start=1):

        print(f"\n  ── {sparsity} (iter {start}) ──")

        # ── sparse activations ────────────────────────────────────────────────
        sparse_activ_path = f"/workspace/CCE_NLI/{model}/activations/{method}/{run}/{start}_Pruning_Iter/final_layer_activations.pkl"
        if not os.path.exists(sparse_activ_path):
            print(f"    Sparse activations not found, skipping.")
            continue
        with open(sparse_activ_path, 'rb') as f:
            sparse_activations = torch.tensor(pickle.load(f))
            

        # ── sparse explanations ───────────────────────────────────────────────
        sparse_expls_root = os.path.join(run_path, 'Expls', sparsity)
        sparse_expls = {}
        for cl in [1, 2, 3]:
            e, _ = load_csv_data(os.path.join(sparse_expls_root, f'Cluster{cl}IOUS1024N.csv'))
            sparse_expls[cl] = e

        # ── sparse masks ──────────────────────────────────────────────────────
        sparse_mask_path = os.path.join(run_path, 'Masks', sparsity)
        sparse_masks = {}
        sparse_dead={}
        for cl in [1, 2, 3]:
            sparse_masks[cl] = get_mask(sparse_mask_path, cluster=cl).t()
            sparse_dead[cl] = get_subactiv(sparse_masks[cl], cluster=cl)


        # ── metrics accumulators for this (run, iter) ─────────────────────────
        cluster_holistic     = {1: 0, 2: 0, 3: 0}
        concepts_not_encoded = {1: 0, 2: 0, 3: 0}
        cluster_per_neuron   = {1: 0, 2: 0, 3: 0}
        cluster_counts       = {1: 0, 2: 0, 3: 0}
        concept_hol_all      = 0
        per_neuron_all       = 0
        per_neuron_all_count = 0
        num_samples          = 600

        avg_sparse = 0
        avg_dense  = 0
        percent_of_dense_firing_concepts_preserved_in_sparse = {1: 0, 2: 0, 3: 0}
        for sample in range(600):
            sparse_neurons_random = set()
            all_sparse_neurons    = set()
            all_dense_neurons     = set()
            sparse_neurons_dict   = {}
            dense_neurons_dict    = {}

            for cl in [1, 2, 3]:
                dense_mask, d_expls = clusters_dense[cl]
                s_mask  = sparse_masks[cl]
                s_expls = sparse_expls[cl]

                sparse_neurons = find_activating_neurons_at_cluster(
                    s_mask.t()[sample], sparse_activations[sample], sparse_dead
                )
                dense_neurons = find_activating_neurons_at_cluster(
                    dense_mask.t()[sample], dense_activations[sample], dense_dead
                )
          
                import random
                #sparse_neurons=dense_neurons
                avg = 0
                for _ in range(50):
                    
                    complement    = list(set(range(1024)) - set(sparse_neurons))
                    random_neurons = random.sample(complement, len(sparse_neurons))

                    
                    missed = 0
                    neurons_that_reconstruct = set()
                    missedcsp=[]
                    if len(dense_neurons) > 0:
                        d_union = set().union(*[s_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                        random_union = set().union(*[s_expls[n] for n in random_neurons]) if random_neurons else set()

                        avg += len(d_union & random_union) / len(random_union | d_union)

                    else:
                        print("NO dense neurons fire for this sample")
                    #print(f"For sample {sample} at cluster {cl}, {d_union} was recog in dense and {s_union} in sparse.\n{missedcsp} is missed ({100*len(missedcsp)/len(d_union)}%)" )
                percent_of_dense_firing_concepts_preserved_in_sparse[cl]  += avg/50
                    
                        
                

        for cl in [1, 2, 3]:
            r['cluster_holistic'][cl].append(
                percent_of_dense_firing_concepts_preserved_in_sparse[cl] / num_samples
            )
            print(cl, percent_of_dense_firing_concepts_preserved_in_sparse[cl] / num_samples)

# ── print averaged results ────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("AVERAGED RESULTS ACROSS RUNS")
print(f"{'='*60}")
print(avg_sparse / 600)
print(avg_dense  / 600)

for idx in iter_indices:
    r     = results_across_runs[idx]
    label = r['sparsity_labels'][0] if r['sparsity_labels'] else f"iter {idx}"
    print(f"\nIteration {idx}  (e.g. {label},  n={len(r['sparsity_labels'])} runs)")

    for cl in [1, 2, 3]:
        hol_vals  = r['cluster_holistic'][cl]
        
        print(f"  Cluster {cl}  holistic:     {np.mean(hol_vals):.4f} ± {np.std(hol_vals):.4f}  (n={len(hol_vals)})")
    


Found runs: ['Run0.25_5']
  Run0.25_5: ['25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
Max iterations: 5

RUN: Run0.25_5

  ── 25.0%Pruned (iter 1) ──


/tmp/ipykernel_160852/3838827412.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_160852/3838827412.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


1 0.5436600948844968
2 0.43824657968841507
3 0.27488907878926755

  ── 43.75%Pruned (iter 2) ──


/tmp/ipykernel_160852/3838827412.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_160852/3838827412.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


1 0.5059610465254432
2 0.471602885482076
3 0.24239444680571065

  ── 57.812%Pruned (iter 3) ──


...

/tmp/ipykernel_160852/3838827412.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_160852/3838827412.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


1 0.32772952701490626
2 0.32011179823753705
3 0.21434413726718765

  ── 76.27%Pruned (iter 5) ──


/tmp/ipykernel_160852/3838827412.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_160852/3838827412.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


1 0.3276238140452054
2 0.3116947902079537
3 0.23613553011674568

AVERAGED RESULTS ACROSS RUNS
0.0
0.0

Iteration 1  (e.g. iter 1,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 3  holistic:     nan ± nan  (n=0)

Iteration 2  (e.g. iter 2,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 3  holistic:     nan ± nan  (n=0)

Iteration 3  (e.g. iter 3,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 3  holistic:     nan ± nan  (n=0)

Iteration 4  (e.g. iter 4,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 3  holistic:     nan ± nan  (n=0)

Iteration 5  (e.g. iter 5,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 3  holistic:     nan ± nan  (n=0)


/usr/local/lib/python3.8/dist-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [39]:
import os
import pickle
import torch
from collections import defaultdict
import numpy as np

model  = 'LLAMA_WANDA'
method = 'wanda'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── discover all Run* directories ─────────────────────────────────────────────
exp_root = f"/workspace/CCE_NLI/{model}/exp/{method}/"
run_dirs = sorted([
    d for d in os.listdir(exp_root)
    if os.path.isdir(os.path.join(exp_root, d))
    and d.startswith('Run')
    and '4cl' not in d
])
print(f"Found runs: {run_dirs}")

# ── collect sparsity levels PER RUN (so index alignment is run-local) ─────────
run_sparsity_levels = {}
for run in run_dirs:
    expls_dir = os.path.join(exp_root, run, 'Expls')
    levels = []
    if os.path.exists(expls_dir):
        for s in sorted(os.listdir(expls_dir)):
            if '.ipynb' not in s and '0.0%Pruned' not in s:
                levels.append(s)
    run_sparsity_levels[run] = levels
    print(f"  {run}: {levels}")

max_iters    = max(len(v) for v in run_sparsity_levels.values())
iter_indices = list(range(1, max_iters + 1))
print(f"Max iterations: {max_iters}")

# ── accumulators: iter_index -> metric lists across runs ──────────────────────
results_across_runs = {
    idx: {
        'cluster_holistic':     defaultdict(list),
        'cluster_per_neuron':   defaultdict(list),
        'concepts_not_encoded': defaultdict(list),
        'percent_of_dense_firing_concepts_preserved_in_sparse': defaultdict(list),
        'concept_hol_all':      [],
        'per_neuron_all':       [],
        'sparsity_labels':      [],   # one entry per run for display
    }
    for idx in iter_indices
}

dense_run_dir  = 'Run0.25_5'
dense_run_path = os.path.join(f"/workspace/CCE_NLI/LLAMA/exp/lottery_ticket", dense_run_dir)

# ── dense activations ─────────────────────────────────────────────────────────
dense_activ_path = f"/workspace/CCE_NLI/LLAMA/activations/lottery_ticket/{dense_run_dir}/0_Pruning_Iter/final_layer_activations.pkl"
if not os.path.exists(dense_activ_path):
    print(f"  Dense activations not found, skipping.")

with open(dense_activ_path, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls_root = os.path.join(dense_run_path, 'Expls', '0.0%Pruned')
dense_expls = {}
for cl in [1, 2, 3]:
    e, _ = load_csv_data(os.path.join(dense_expls_root, f'Cluster{cl}IOUS1024N.csv'))

    dense_expls[cl] = e

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_run_path, 'Masks', '0.0%Pruned')
dense_masks = {}
dense_dead={}
for cl in [1, 2, 3]:
    dense_masks[cl] = get_mask(dense_mask_path, cluster=cl).t()
    dense_dead[cl]=get_subactiv(dense_masks[cl], cl)

# ── dense model weights ───────────────────────────────────────────────────────
dense_weights_path = f"/workspace/CCE_NLI/LLAMA/models/lottery_ticket/{dense_run_dir}/0_Pruning_Iter/model_best.pth"
if not os.path.exists(dense_weights_path):
    print(f"  Dense weights not found, skipping.")

dense_model_finallayerweights = torch.load(
    dense_weights_path, map_location=device
)['state_dict']['mlp.3.weight']

clusters_dense = {cl: (dense_masks[cl], dense_expls[cl]) for cl in [1, 2, 3]}

# ── main loop over runs ───────────────────────────────────────────────────────
for run in run_dirs:
    print(f"\n{'='*60}")
    print(f"RUN: {run}")
    print(f"{'='*60}")

    run_path = os.path.join(exp_root, run)

    # iterate over THIS RUN's own sorted sparsity levels — index is the key
    for start, sparsity in enumerate(run_sparsity_levels[run], start=1):

        print(f"\n  ── {sparsity} (iter {start}) ──")

        # ── sparse activations ────────────────────────────────────────────────
        sparse_activ_path = f"/workspace/CCE_NLI/{model}/activations/{method}/{run}/{start}_Pruning_Iter/final_layer_activations.pkl"
        if not os.path.exists(sparse_activ_path):
            print(f"    Sparse activations not found, skipping.")
            continue
        with open(sparse_activ_path, 'rb') as f:
            sparse_activations = torch.tensor(pickle.load(f))
            

        # ── sparse explanations ───────────────────────────────────────────────
        sparse_expls_root = os.path.join(run_path, 'Expls', sparsity)
        sparse_expls = {}
        for cl in [1, 2, 3]:
            e, _ = load_csv_data(os.path.join(sparse_expls_root, f'Cluster{cl}IOUS1024N.csv'))
            sparse_expls[cl] = e

        # ── sparse masks ──────────────────────────────────────────────────────
        sparse_mask_path = os.path.join(run_path, 'Masks', sparsity)
        sparse_masks = {}
        sparse_dead={}
        for cl in [1, 2, 3]:
            sparse_masks[cl] = get_mask(sparse_mask_path, cluster=cl).t()
            sparse_dead[cl] = get_subactiv(sparse_masks[cl], cluster=cl)

        # ── sparse model weights ──────────────────────────────────────────────
        sparse_weights_path = f"/workspace/CCE_NLI/{model}/models/{method}/{run}/{start}_Pruning_Iter/model_best.pth"
        if not os.path.exists(sparse_weights_path):
            print(f"    Sparse weights not found, skipping.")
            continue
        sparse_model_finallayerweights = torch.load(
            sparse_weights_path, map_location=device
        )['state_dict']['mlp.3.weight']

        if method == 'CoFi':
            print(f"Sparsity: {sparsity}\nStart: {start}")
            zs_path = f"/workspace/CCE_NLI/{model}/models/{method}/{run}/{start}_Pruning_Iter/zs.pt"
            if os.path.exists(zs_path):
                zs = torch.load(zs_path, map_location=device)
                sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
                    zs['final_mlp_hidden_z'].to(device).squeeze()
                )
                sparse_model_finallayerweights = sparse_model_finallayerweights[sparse_model_finallayerweights!=0]
                sparse_model_finallayerweights = sparse_model_finallayerweights.reshape((3, sparse_model_finallayerweights.shape[0]//3))

        # ── metrics accumulators for this (run, iter) ─────────────────────────
        cluster_holistic     = {1: 0, 2: 0, 3: 0}
        concepts_not_encoded = {1: 0, 2: 0, 3: 0}
        cluster_per_neuron   = {1: 0, 2: 0, 3: 0}
        cluster_counts       = {1: 0, 2: 0, 3: 0}
        concept_hol_all      = 0
        per_neuron_all       = 0
        per_neuron_all_count = 0
        num_samples          = 600

        avg_sparse = 0
        avg_dense  = 0
        percent_of_dense_firing_concepts_preserved_in_sparse = {1: 0, 2: 0, 3: 0}
        for sample in range(600):
            sparse_neurons_random = set()
            all_sparse_neurons    = set()
            all_dense_neurons     = set()
            sparse_neurons_dict   = {}
            dense_neurons_dict    = {}

            for cl in [1, 2, 3]:
                dense_mask, d_expls = clusters_dense[cl]
                s_mask  = sparse_masks[cl]
                s_expls = sparse_expls[cl]

                sparse_neurons = find_activating_neurons_at_cluster(
                    s_mask.t()[sample], sparse_activations[sample], sparse_dead
                )
                dense_neurons = find_activating_neurons_at_cluster(
                    dense_mask.t()[sample], dense_activations[sample], dense_dead
                )
                #sparse_neurons=dense_neurons
                '''import random
                complement    = list(set(range(1024)) - set(sparse_neurons) - set(sparse_neurons_random))
                sparse_neurons = random.sample(complement, len(sparse_neurons))
                sparse_neurons_random |= set(sparse_neurons)'''

                sparse_neurons_dict[cl] = sparse_neurons
                dense_neurons_dict[cl]  = dense_neurons
                all_sparse_neurons     |= set(sparse_neurons)
                all_dense_neurons      |= set(dense_neurons)

                missed = 0
                neurons_that_reconstruct = set()
                missedcsp=[]
                if len(dense_neurons) > 0:
                    d_union = set().union(*[d_expls[n] for n in dense_neurons]) if dense_neurons else set()
                    s_union = set().union(*[s_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                    
                    all_sprase_concepts = set().union(*[s_expls.get(n,set()) for n in range(1024)])
                    #percent_of_dense_firing_concepts_preserved_in_sparse[cl] += len(d_union & all_sprase_concepts) / len(d_union)
                    #print(cl, sample, len(d_union & all_sprase_concepts) / len(d_union))

                    if len(d_union) > 0:
                        for d in list(d_union):
                            no_neurons_encoded = True
                            for s in sparse_neurons:
                                if d in s_expls[s]:
                                    neurons_that_reconstruct.add(s)
                                    no_neurons_encoded = False
                            if no_neurons_encoded:
                                missedcsp.append(d)
                                missed += 1
                        #print(f"Missed {missed} concpets {d_union}")

                    
                        #print(f"Missed : {missed} concepts out of a total of {len(list(d_union))}")
                        #if missed / len(list(d_union)) < 1:
                            #print(f"Found {neurons_that_reconstruct} out of {len(sparse_neurons)}")
                        if len(sparse_neurons)>0:
                            cluster_holistic[cl]     += len(neurons_that_reconstruct) / len(sparse_neurons)
                        concepts_not_encoded[cl] += missed / len(list(d_union))
                    #else:
                        #print(cl, "No active neuron fires for this sample")
                #else:
                    #rint("NO dense neurons fire for this sample")
                #print(f"For sample {sample} at cluster {cl}, {d_union} was recog in dense and {s_union} in sparse.\n{missedcsp} is missed ({100*len(missedcsp)/len(d_union)}%)" )
                             
                    
                        
                    
              
        # ── normalize over samples and store under iteration index ────────────
        r = results_across_runs[start]
        r['sparsity_labels'].append(sparsity)

        for cl in [1, 2, 3]:
            r['cluster_holistic'][cl].append(
                cluster_holistic[cl] / num_samples
            )
            
            r['concepts_not_encoded'][cl].append(
                concepts_not_encoded[cl] / num_samples
            )
            print(r['concepts_not_encoded'][cl])
           
            

       
# ── print averaged results ────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("AVERAGED RESULTS ACROSS RUNS")
print(f"{'='*60}")
print(avg_sparse / 600)
print(avg_dense  / 600)

for idx in iter_indices:
    r     = results_across_runs[idx]
    label = r['sparsity_labels'][0] if r['sparsity_labels'] else f"iter {idx}"
    print(f"\nIteration {idx}  (e.g. {label},  n={len(r['sparsity_labels'])} runs)")

    for cl in [1, 2, 3]:
        hol_vals  = r['cluster_holistic'][cl]
        cne_vals  = r['concepts_not_encoded'][cl]
       
        print(f"  Cluster {cl}  holistic:     {np.mean(hol_vals):.4f} ± {np.std(hol_vals):.4f}  (n={len(hol_vals)})")
        print(f"  Cluster {cl}  dense missed: {np.mean(cne_vals):.4f} ± {np.std(cne_vals):.4f}")
   

Found runs: ['Run0.25_5']
  Run0.25_5: ['25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
Max iterations: 5

RUN: Run0.25_5

  ── 25.0%Pruned (iter 1) ──


/tmp/ipykernel_160852/3838827412.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_160852/3838827412.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


[0.2998970211509637]
[0.22033612088414622]
[0.216824306773944]

  ── 43.75%Pruned (iter 2) ──
    Sparse activations not found, skipping.

  ── 57.812%Pruned (iter 3) ──
    Sparse activations not found, skipping.

  ── 68.359%Pruned (iter 4) ──
    Sparse activations not found, skipping.

  ── 76.27%Pruned (iter 5) ──
    Sparse activations not found, skipping.

AVERAGED RESULTS ACROSS RUNS
0.0
0.0

Iteration 1  (e.g. 25.0%Pruned,  n=1 runs)
  Cluster 1  holistic:     0.9994 ± 0.0000  (n=1)
  Cluster 1  dense missed: 0.2999 ± 0.0000
  Cluster 2  holistic:     0.9981 ± 0.0000  (n=1)
  Cluster 2  dense missed: 0.2203 ± 0.0000
  Cluster 3  holistic:     0.8567 ± 0.0000  (n=1)
  Cluster 3  dense missed: 0.2168 ± 0.0000

Iteration 2  (e.g. iter 2,  n=0 runs)
  Cluster 1  holistic:     nan ± nan  (n=0)
  Cluster 1  dense missed: nan ± nan
  Cluster 2  holistic:     nan ± nan  (n=0)
  Cluster 2  dense missed: nan ± nan
  Cluster 3  holistic:     nan ± nan  (n=0)
  Cluster 3  dense missed: na

/usr/local/lib/python3.8/dist-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/usr/local/lib/python3.8/dist-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [37]:
import pandas as pd

data = {
    "sparsity": [25.0, 43.75, 57.812, 68.359, 76.27],
    "c1": [0.27944000283226156, 0.3275875115736209, 0.3905495482341055, 0.4758009229513293, 0.4763497148242333],
    "c2": [0.27224329032290745, 0.3181343161936467, 0.41183174653507254, 0.5324575921871897, 0.6034267949158866],
    "c3": [0.2998001931188326, 0.4970216610664077, 0.6798444143557499, 0.6910491668121459, 0.6968933500745403],
}

df = pd.DataFrame(data)
print(df)

   sparsity        c1        c2        c3
0    25.000  0.279440  0.272243  0.299800
1    43.750  0.327588  0.318134  0.497022
2    57.812  0.390550  0.411832  0.679844
3    68.359  0.475801  0.532458  0.691049
4    76.270  0.476350  0.603427  0.696893


concept alignment (clusterwise per neuron and holistiic, and across cluster per neuron and holisic

In [358]:
import os
import pickle
import torch
from collections import defaultdict
import numpy as np

model  = 'BERT'
method = 'lottery_ticket'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── discover all Run* directories ─────────────────────────────────────────────
exp_root = f"/workspace/CCE_NLI/{model}/exp/{method}/"
run_dirs = sorted([
    d for d in os.listdir(exp_root)
    if os.path.isdir(os.path.join(exp_root, d))
    and d.startswith('Run')
    and '4cl' not in d
])
print(f"Found runs: {run_dirs}")

# ── collect all sparsity levels (union across all runs) ───────────────────────
sparsity_levels = set()
for run in run_dirs:
    expls_dir = os.path.join(exp_root, run, 'Expls')
    if os.path.exists(expls_dir):
        for s in os.listdir(expls_dir):
            if '.ipynb' not in s and '0.0%Pruned' not in s:
                sparsity_levels.add(s)
sparsity_levels = sorted(sparsity_levels)
print(f"Sparsity levels: {sparsity_levels}")

# ── accumulators: sparsity -> list of metric values across runs ───────────────
# each entry is averaged over 600 samples within a run, then we average over runs
results_across_runs = {
    sparsity: {
        'cluster_holistic':  defaultdict(list),   # cl -> [val_run0, val_run1, ...]
        'cluster_per_neuron': defaultdict(list),
        'concept_hol_all':   [],
        'per_neuron_all':    [],
    }
    for sparsity in sparsity_levels
}
dense_run_dir='Run0.25_5'
dense_run_path =  os.path.join(f"/workspace/CCE_NLI/{model}/exp/lottery_ticket", 'Run0.25_5')
# ── dense activations for this run ────────────────────────────────────────
dense_activ_path = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/{dense_run_dir}/0_Pruning_Iter/final_layer_activations.pkl"
if not os.path.exists(dense_activ_path):
    print(f"  Dense activations not found for {run}, skipping.")

with open(dense_activ_path, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────
dense_expls_root = os.path.join(dense_run_path, 'Expls', '0.0%Pruned')
dense_expls = {}
for cl in [1, 2, 3]:
    e, _ = load_csv_data(os.path.join(dense_expls_root, f'Cluster{cl}IOUS1024N.csv'))
    dense_expls[cl] = e

# ── dense masks ───────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_run_path, 'Masks', '0.0%Pruned')
dense_masks = {}
for cl in [1, 2, 3]:
    dense_masks[cl] = get_mask(dense_mask_path, cluster=cl).t()
#dense_dead = get_subactiv(dense_mask_path, cluster=1)

# ── dense model weights ───────────────────────────────────────────────────
dense_weights_path = f"/workspace/CCE_NLI/{model}/models/lottery_ticket/{dense_run_dir}/0_Pruning_Iter/model_best.pth"
if not os.path.exists(dense_weights_path):
    print(f"  Dense weights not found for {run}, skipping.")
    

dense_model_finallayerweights = torch.load(
    dense_weights_path, map_location=device
)['state_dict']['mlp.3.weight']

clusters_dense = {cl: (dense_masks[cl], dense_expls[cl]) for cl in [1, 2, 3]}

# ── main loop over runs ───────────────────────────────────────────────────────
for run in run_dirs[:1]:
    print(f"\n{'='*60}")
    print(f"RUN: {run}")
    print(f"{'='*60}")

    run_path = os.path.join(exp_root, run)

    # ── sparsity loop for this run ────────────────────────────────────────────
    for start, sparsity in enumerate(sparsity_levels, start=1):

        print(f"\n  ── {sparsity} (iter {start}) ──")

        # ── sparse activations ────────────────────────────────────────────────
        sparse_activ_path = f"/workspace/CCE_NLI/{model}/activations/{method}/{run}/{start}_Pruning_Iter/final_layer_activations.pkl"
        if not os.path.exists(sparse_activ_path):
            print(f"    Sparse activations not found, skipping.")
            continue
        with open(sparse_activ_path, 'rb') as f:
            sparse_activations = torch.tensor(pickle.load(f))
        
        
        # ── sparse explanations ───────────────────────────────────────────────
        sparse_expls_root = os.path.join(run_path, 'Expls', sparsity)
        sparse_expls = {}
        for cl in [1, 2, 3]:
            e, _ = load_csv_data(os.path.join(sparse_expls_root, f'Cluster{cl}IOUS1024N.csv'))
            sparse_expls[cl] = e

        # ── sparse masks ──────────────────────────────────────────────────────
        sparse_mask_path = os.path.join(run_path, 'Masks', sparsity)
        sparse_masks = {}
        for cl in [1, 2, 3]:
            sparse_masks[cl] = get_mask(sparse_mask_path, cluster=cl).t()
            sparse_dead = get_subactiv(sparse_masks[cl], cluster=cl)
            

        # ── sparse model weights ──────────────────────────────────────────────
        sparse_weights_path = f"/workspace/CCE_NLI/{model}/models/{method}/{run}/{start}_Pruning_Iter/model_best.pth"
        if not os.path.exists(sparse_weights_path):
            print(f"    Sparse weights not found, skipping.")
            continue
        sparse_model_finallayerweights = torch.load(
            sparse_weights_path, map_location=device
        )['state_dict']['mlp.3.weight']

        if method == 'CoFi':
            zs_path = f"/workspace/CCE_NLI/{model}/models/{method}/{run}/{start}_Pruning_Iter/zs.pt"
            if os.path.exists(zs_path):
                zs = torch.load(zs_path, map_location=device)
                sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
                    zs['final_mlp_hidden_z'].to(device)
                )

        # ── metrics accumulators for this (run, sparsity) ─────────────────────
        cluster_holistic   = {1: 0, 2: 0, 3: 0}
        cluster_per_neuron = {1: 0, 2: 0, 3: 0}
        cluster_counts     = {1: 0, 2: 0, 3: 0}
        concept_hol_all    = 0
        per_neuron_all     = 0
        per_neuron_all_count = 0
        num_samples        = 300
        sparse_neurons_random=set()
        for sample in range(300):

            all_sparse_neurons = set()
            all_dense_neurons  = set()
            sparse_neurons_dict = {}
            dense_neurons_dict  = {}
            
            for cl in [1, 2, 3]:
                dense_mask, d_expls = clusters_dense[cl]
                s_mask  = sparse_masks[cl]
                s_expls = sparse_expls[cl]
                
                sparse_neurons = find_highest_activating_neurons_at_cluster(
                    s_mask.t()[sample], sparse_activations[sample],
                    sparse_model_finallayerweights, sparse_dead
                )
                dense_neurons = find_highest_activating_neurons_at_cluster(
                    dense_mask.t()[sample], dense_activations[sample],
                    dense_model_finallayerweights, dense_dead
                )
                
                #print(torch.where(sparse_activations[sample]>0,1,0).sum(), torch.where(s_mask.t()[sample]>0,1,0).sum())
                #print(len(set(sparse_neurons) & set(dense_neurons)))
                
                import random
                complement = list(set(range(1024)) - set(sparse_neurons) - set(dense_neurons) - set(sparse_neurons_random) )
                sparse_neurons = random.sample(complement, len(sparse_neurons))
                sparse_neurons_random |= set(sparse_neurons)
                sparse_neurons_dict[cl] = sparse_neurons
                
                
                dense_neurons_dict[cl]  = dense_neurons
                #assert len(set(dense_neurons) - set(sparse_neurons)) == 0 and len(set(sparse_neurons) - set(dense_neurons))==0
                all_sparse_neurons |= set(sparse_neurons)
                all_dense_neurons  |= set(dense_neurons)
                
                # per-cluster holistic
                if len(dense_neurons) > 0:
                    s_union = set().union(*[s_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                    d_union = set().union(*[d_expls[n] for n in dense_neurons])  if dense_neurons  else set()
                    if len(d_union) > 0:
                        cluster_holistic[cl] += len(s_union & d_union) / len(s_union | d_union)

                # per-cluster per-neuron
                for s, d in zip(set(sparse_neurons), set(dense_neurons)):
                    s_concepts = s_expls[s]
                    d_concepts = d_expls[d]
                    if len(d_concepts) == 0:
                        continue
                    cluster_per_neuron[cl] += len(s_concepts & d_concepts) / len(s_concepts| d_concepts)
                    
                    cluster_counts[cl] += 1

            # all-cluster holistic
            s_union_all = set()
            d_union_all = set()
            for cl in [1, 2, 3]:
                #print(len(all_dense_neurons))
                s_union_all |= set().union(*[sparse_expls[cl][n] for n in sparse_neurons_dict[cl]]) if sparse_neurons_dict[cl] else set()
           
                d_union_all |= set().union(*[clusters_dense[cl][1][n] for n in dense_neurons_dict[cl]]) if dense_neurons_dict[cl] else set()
                
            
            if len(d_union_all) > 0:
                #print(f'Number of concepts in sparse union: {len(s_union_all)}\n# in dense union:{len(d_union_all)}\nIntersection:{len(s_union_all & d_union_all)}\nUnion : {len(s_union_all | d_union_all)}\nIoU: {len(s_union_all & d_union_all) / len(s_union_all | d_union_all)}\nPreservation: {len(s_union_all & d_union_all)/len( d_union_all)}')
                
                concept_hol_all += len(s_union_all & d_union_all) / len(s_union_all | d_union_all)
                #print(sparsity, concept_hol_all)
               

            # all-cluster per-neuron
            for s, d in zip(all_sparse_neurons, all_dense_neurons):
                s_concepts = set()
                d_concepts = set()
                for cl in [1, 2, 3]:
                    if s in sparse_expls[cl]:
                        s_concepts |= sparse_expls[cl][s]
                    if d in clusters_dense[cl][1]:
                        d_concepts |= clusters_dense[cl][1][d]
                if len(d_concepts) == 0:
                    continue
                per_neuron_all += len(s_concepts & d_concepts) / len(s_concepts | d_concepts)
                per_neuron_all_count += 1

        # ── normalize over samples ────────────────────────────────────────────
        for cl in [1, 2, 3]:
            results_across_runs[sparsity]['cluster_holistic'][cl].append(
                cluster_holistic[cl] / num_samples
            )
            results_across_runs[sparsity]['cluster_per_neuron'][cl].append(
                cluster_per_neuron[cl] / cluster_counts[cl] if cluster_counts[cl] > 0 else 0
            )

        results_across_runs[sparsity]['concept_hol_all'].append(
            concept_hol_all / num_samples
        )
        results_across_runs[sparsity]['per_neuron_all'].append(
            per_neuron_all / per_neuron_all_count if per_neuron_all_count > 0 else 0
        )

# ── print averaged results ────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("AVERAGED RESULTS ACROSS RUNS")
print(f"{'='*60}")

for sparsity in sparsity_levels:
    r = results_across_runs[sparsity]
    print(f"\n{sparsity}")

    for cl in [1, 2, 3]:
        hol_vals = r['cluster_holistic'][cl]
        pn_vals  = r['cluster_per_neuron'][cl]
        print(f"  Cluster {cl}  holistic:   {np.mean(hol_vals):.4f} ± {np.std(hol_vals):.4f}  (n={len(hol_vals)} runs)")
        print(f"  Cluster {cl}  per-neuron: {np.mean(pn_vals):.4f}  ± {np.std(pn_vals):.4f}")

    hol_all = r['concept_hol_all']
    pn_all  = r['per_neuron_all']
    print(f"  All clusters holistic:   {np.mean(hol_all):.4f} ± {np.std(hol_all):.4f}")
    print(f"  All clusters per-neuron: {np.mean(pn_all):.4f}  ± {np.std(pn_all):.4f}")

Found runs: ['Run0.25_5', 'Run0.25_6', 'Run0.25_7']
Sparsity levels: ['25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']

RUN: Run0.25_5

  ── 25.0%Pruned (iter 1) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)



  ── 43.75%Pruned (iter 2) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)



  ── 57.812%Pruned (iter 3) ──


...


  ── 68.359%Pruned (iter 4) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)



  ── 76.27%Pruned (iter 5) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)



AVERAGED RESULTS ACROSS RUNS

25.0%Pruned
  Cluster 1  holistic:   0.4735 ± 0.0000  (n=1 runs)
  Cluster 1  per-neuron: 0.1416  ± 0.0000
  Cluster 2  holistic:   0.4695 ± 0.0000  (n=1 runs)
  Cluster 2  per-neuron: 0.0930  ± 0.0000
  Cluster 3  holistic:   0.3370 ± 0.0000  (n=1 runs)
  Cluster 3  per-neuron: 0.0763  ± 0.0000
  All clusters holistic:   0.4902 ± 0.0000
  All clusters per-neuron: 0.0896  ± 0.0000

43.75%Pruned
  Cluster 1  holistic:   0.4528 ± 0.0000  (n=1 runs)
  Cluster 1  per-neuron: 0.1398  ± 0.0000
  Cluster 2  holistic:   0.4439 ± 0.0000  (n=1 runs)
  Cluster 2  per-neuron: 0.0951  ± 0.0000
  Cluster 3  holistic:   0.3167 ± 0.0000  (n=1 runs)
  Cluster 3  per-neuron: 0.0752  ± 0.0000
  All clusters holistic:   0.4755 ± 0.0000
  All clusters per-neuron: 0.0885  ± 0.0000

57.812%Pruned
  Cluster 1  holistic:   0.4580 ± 0.0000  (n=1 runs)
  Cluster 1  per-neuron: 0.1408  ± 0.0000
  Cluster 2  holistic:   0.4234 ± 0.0000  (n=1 runs)
  Cluster 2  per-neuron: 0.0906  ± 0

In [ ]:
concept preservatoin (clusterwise per neuron and holistiic, and across cluster per neuron and holisic

In [356]:
model = 'BERT'
method = 'lottery_ticket'


print(f"{model} {method} CLUSTER {c}")

# ── paths ─────────────────────────────────────────────────────────────────────
path_to_experiment = os.path.join("/workspace/CCE_NLI", model, 'exp', method, 'Run0.25_5')
dense_path_root    = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root   = path_to_experiment

# ── dense activations ─────────────────────────────────────────────────────────
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls1, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster1IOUS1024N.csv'))
dense_expls2, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster2IOUS1024N.csv'))
dense_expls3, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster3IOUS1024N.csv'))

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_path_root, 'Masks', '0.0%Pruned')
dense_mask1 = get_mask(dense_mask_path, cluster=1).t()
dense_mask2 = get_mask(dense_mask_path, cluster=2).t()
dense_mask3 = get_mask(dense_mask_path, cluster=3).t()
dense_dead  = [] #get_subactiv(dense_mask_path, cluster=c)
print(f"Dense masks loaded — shapes: {dense_mask1.shape}, {dense_mask2.shape}, {dense_mask3.shape}")

# ── dense model weights ───────────────────────────────────────────────────────
dense_model_finallayerweights = torch.load(
    f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth',
    map_location=device
)['state_dict']['mlp.3.weight']

# ── sparsity loop ─────────────────────────────────────────────────────────────
start = 1

clusters_dense = {
    1: (dense_mask1, dense_expls1),
    2: (dense_mask2, dense_expls2),
    3: (dense_mask3, dense_expls3),
}

start = 1

for i, sparsity in enumerate(sorted(os.listdir(os.path.join(path_to_experiment, 'Expls')))):

    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity:
        continue

    print(f"\n── {sparsity} (Pruning iter {start}) ──")

    # ── sparse activations ─────────────────────────────────────────
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))

    # ── sparse explanations ────────────────────────────────────────
    sparse_expls1, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster1IOUS1024N.csv'))
    #sparse_expls1, _ = load_csv_data(os.path.join('/workspace/CCE_NLI/BERT/exp/untrained/Expls/Cluster1IOUS1024N.csv'))
    
    sparse_expls2, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster2IOUS1024N.csv'))
    #sparse_expls2, _ = load_csv_data(os.path.join('/workspace/CCE_NLI/BERT/exp/untrained/Expls/Cluster2IOUS1024N.csv'))
    
    sparse_expls3, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster3IOUS1024N.csv'))
    #sparse_expls3, _ = load_csv_data(os.path.join('/workspace/CCE_NLI/BERT/exp/untrained/Expls/Cluster3IOUS1024N.csv'))
    

    clusters_sparse = {
        1: sparse_expls1,
        2: sparse_expls2,
        3: sparse_expls3,
    }

    # ── sparse masks ───────────────────────────────────────────────
    sparse_mask_path = os.path.join(sparse_path_root, 'Masks', sparsity)
    sparse_mask1 = get_mask(sparse_mask_path, cluster=1).t()
    sparse_mask2 = get_mask(sparse_mask_path, cluster=2).t()
    sparse_mask3 = get_mask(sparse_mask_path, cluster=3).t()
    sparse_dead  = [] #get_subactiv(sparse_mask_path, cluster=c)

    
    redundnacy_files = [pd.read_csv(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{sparsity}/Cluster1Redundance.csv'),
                       pd.read_csv(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{sparsity}/Cluster2Redundance.csv'),
                       pd.read_csv(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{sparsity}/Cluster3Redundance.csv')]
    clusters_sparse_masks = {
        1: sparse_mask1,
        2: sparse_mask2,
        3: sparse_mask3,
    }

    # ── sparse model weights ───────────────────────────────────────
    sparse_model_finallayerweights = torch.load(
        f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth',
        map_location=device
    )['state_dict']['mlp.3.weight']
    if method == 'CoFi':
        zs = torch.load(
            f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt',
            map_location=device
        )
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
            zs['final_mlp_hidden_z'].to(device)
        )

    # ── metrics init ───────────────────────────────────────────────
    cluster_holistic = {1: 0, 2: 0, 3: 0}
    cluster_per_neuron = {1: 0, 2: 0, 3: 0}
    cluster_counts = {1: 0, 2: 0, 3: 0}

    concept_hol_dif_all = 0
    per_neuron_all = 0
    per_neuron_all_count = 0
    
    avg_all_sparse =0
    avg_all_dense=0
    # ── main loop ─────────────────────────────────────────────────
    for sample in range(300):

        all_sparse_neurons = set()
        all_dense_neurons = set()

        sparse_neurons_dict = {}
        dense_neurons_dict = {}

        for cl in [1, 2, 3]:
            

            dense_mask, dense_expls = clusters_dense[cl]
            sparse_mask = clusters_sparse_masks[cl]
       
            sparse_expls = clusters_sparse[cl]
        
            sparse_neurons = find_highest_activating_neurons_at_cluster(
                sparse_mask.t()[sample],
                sparse_activations[sample],
                sparse_model_finallayerweights,
                sparse_dead
            )

            dense_neurons = find_highest_activating_neurons_at_cluster(
                dense_mask.t()[sample],
                dense_activations[sample],
                dense_model_finallayerweights,
                dense_dead
            )
            '''import random

            complement = list(set(range(1024)) - set(sparse_neurons) - set(dense_neurons))
            sparse_neurons = random.sample(complement, len(sparse_neurons))'''
            #sparse_neurons=dense_neurons
            sparse_neurons_dict[cl] = sparse_neurons
            dense_neurons_dict[cl] = dense_neurons
            #print(len(sparse_neurons), len(set(sparse_neurons)&set(dense_neurons)), len(dense_neurons))
            all_sparse_neurons |= set(sparse_neurons)
            all_dense_neurons |= set(dense_neurons)

            # ── per-cluster holistic ──
            if len(dense_neurons) > 0:
                s_union = set().union(*[sparse_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                d_union = set().union(*[clusters_dense[cl][1][n] for n in dense_neurons]) if dense_neurons else set()

                if len(s_union_all) > 0:
                    
                    #print(cl, {n:redundnacy_files[cl][n] for n in s_union & d_union})
                    cluster_holistic[cl] += len(s_union & d_union) / len(s_union | d_union)

            # ── per-cluster per-neuron ──
            for s,d in zip(sparse_neurons, dense_neurons):

                s_concepts = sparse_expls[s]
                d_concepts = clusters_dense[cl][1][d]

                if len(d_concepts) == 0:
                    continue

                overlap = len(s_concepts & d_concepts) / len( s_concepts|d_concepts)

                cluster_per_neuron[cl] += overlap
                cluster_counts[cl] += 1

        # ── all clusters holistic ──
        s_union_all = set()
        d_union_all = set()

        for cl in [1, 2, 3]:
            s_union_all |= set().union(*[clusters_sparse[cl][n] for n in sparse_neurons_dict[cl][:2]]) if sparse_neurons_dict[cl] else set()
            d_union_all |= set().union(*[clusters_dense[cl][1][n] for n in dense_neurons_dict[cl][:2]]) if dense_neurons_dict[cl] else set()

        if len(s_union_all) > 0:
            #print(f"Number of holistic sparse concepts (unique): {len(s_union_all)}nNumber of holistic dense concepts (unique):{len(d_union_all)}\nIntersection: {len(s_union_all & d_union_all) }\nUnion:{}\n IoU: {}")
            if (len(s_union_all & d_union_all) / len(s_union_all | d_union_all)) >1 :
                print(sample)
            
            concept_hol_dif_all += len(s_union_all & d_union_all) / len(s_union_all | d_union_all)

        # ── all clusters per-neuron ──
        avg_all_sparse += len(all_sparse_neurons)
        avg_all_dense += len(all_dense_neurons)
        for s,d in zip(all_sparse_neurons, all_dense_neurons):

            s_concepts = set()
            d_concepts = set()

            for cl in [1, 2, 3]:
                if s in clusters_sparse[cl]:
                    s_concepts |= clusters_sparse[cl][s]
                if d in clusters_dense[cl][1]:
                    d_concepts |= clusters_dense[cl][1][d]

            if len(s_concepts) == 0:
                continue

            overlap = len(s_concepts & d_concepts) / len(s_concepts | d_concepts)

            per_neuron_all += overlap
            per_neuron_all_count += 1

    # ── normalize ─────────────────────────────────────────────────
    num_samples = 300
    print(f"number of sparse neuron covered on average: {avg_all_sparse/num_samples}\nNumber of sparse neuron covered on average : {avg_all_dense/num_samples}")
    cluster_holistic = {cl: cluster_holistic[cl] / num_samples for cl in cluster_holistic}

    cluster_per_neuron = {
        cl: (cluster_per_neuron[cl] / cluster_counts[cl] if cluster_counts[cl] > 0 else 0)
        for cl in cluster_per_neuron
    }

    concept_hol_dif_all /= num_samples
    per_neuron_all = per_neuron_all / per_neuron_all_count if per_neuron_all_count > 0 else 0

    # ── print results ─────────────────────────────────────────────
    print("Cluster Holistic:", cluster_holistic)
    print("Cluster Per-Neuron:", cluster_per_neuron)
    print("All Clusters Holistic:", concept_hol_dif_all)
    print("All Clusters Per-Neuron:", per_neuron_all)

    start += 1

BERT lottery_ticket CLUSTER 1
Dense masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])

── 25.0%Pruned (Pruning iter 1) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


number of sparse neuron covered on average: 328.24333333333334
Number of sparse neuron covered on average : 328.99666666666667
Cluster Holistic: {1: 0.4735238523643612, 2: 0.4695358043026073, 3: 0.33702593087938193}
Cluster Per-Neuron: {1: 0.14005406030022044, 2: 0.09288269671942159, 3: 0.07725609104642908}
All Clusters Holistic: 0.49019309564748315
All Clusters Per-Neuron: 0.08958991652566176

── 43.75%Pruned (Pruning iter 2) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


number of sparse neuron covered on average: 332.11
Number of sparse neuron covered on average : 328.99666666666667
Cluster Holistic: {1: 0.45281717126415544, 2: 0.4439429577463149, 3: 0.316744631268797}
Cluster Per-Neuron: {1: 0.1381961415981591, 2: 0.09747305379234843, 3: 0.07437205813740584}
All Clusters Holistic: 0.4755329310798682
All Clusters Per-Neuron: 0.08849927717263377

── 57.812%Pruned (Pruning iter 3) ──


...

number of sparse neuron covered on average: 335.06
Number of sparse neuron covered on average : 328.99666666666667
Cluster Holistic: {1: 0.45799576283514504, 2: 0.423403372988529, 3: 0.3122982877234064}
Cluster Per-Neuron: {1: 0.14021723005467945, 2: 0.09276795597245059, 3: 0.0731242703094374}
All Clusters Holistic: 0.4633877855476826
All Clusters Per-Neuron: 0.0873497992483014

── 68.359%Pruned (Pruning iter 4) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


number of sparse neuron covered on average: 331.63666666666666
Number of sparse neuron covered on average : 328.99666666666667
Cluster Holistic: {1: 0.45654523473394837, 2: 0.43360364760002595, 3: 0.31207114033781613}
Cluster Per-Neuron: {1: 0.13717660719711303, 2: 0.09398986724378473, 3: 0.07504649858794707}
All Clusters Holistic: 0.4751683415337011
All Clusters Per-Neuron: 0.08663094007163907

── 76.27%Pruned (Pruning iter 5) ──


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)


number of sparse neuron covered on average: 334.24
Number of sparse neuron covered on average : 328.99666666666667
Cluster Holistic: {1: 0.46697770242007514, 2: 0.4508431837056463, 3: 0.3110729592770733}
Cluster Per-Neuron: {1: 0.13635887887379086, 2: 0.09238303104282307, 3: 0.07205713194852285}
All Clusters Holistic: 0.4768482096913086
All Clusters Per-Neuron: 0.08653703400462812


In [141]:
def masked(f):
    return torch.where(f>0,1,0)

neuron alingment pariwise neruon and acors neurons

In [369]:
model = 'BERT'
method = 'lottery_ticket'
c = 1

print(f"{model} {method} CLUSTER {c}")

# ── paths ─────────────────────────────────────────────────────────────────────
path_to_experiment = os.path.join("/workspace/CCE_NLI", model, 'exp', method, 'Run0.25_5')
dense_path_root    = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root   = path_to_experiment

# ── dense activations ─────────────────────────────────────────────────────────
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls1, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster1IOUS1024N.csv'))
dense_expls2, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster2IOUS1024N.csv'))
dense_expls3, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster3IOUS1024N.csv'))

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_path_root, 'Masks', '0.0%Pruned')
dense_mask1 = get_mask(dense_mask_path, cluster=1).t()
dense_mask2 = get_mask(dense_mask_path, cluster=2).t()
dense_mask3 = get_mask(dense_mask_path, cluster=3).t()
dense_dead  = [] #get_subactiv(dense_mask_path, cluster=c)
print(f"Dense masks loaded — shapes: {dense_mask1.shape}, {dense_mask2.shape}, {dense_mask3.shape}")

# ── dense model weights ───────────────────────────────────────────────────────
dense_model_finallayerweights = torch.load(
    f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth',
    map_location=device
)['state_dict']['mlp.3.weight']

# ── sparsity loop ─────────────────────────────────────────────────────────────
start = 1

for i, sparsity in enumerate(sorted(os.listdir(os.path.join(path_to_experiment, 'Expls')))):

    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity:
        continue

    print(f"\n── {sparsity} (Pruning iter {start}) ──")

    # sparse activations
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))

    # sparse explanations
    sparse_expls1, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster1IOUS1024N.csv'))
    sparse_expls2, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster2IOUS1024N.csv'))
    sparse_expls3, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster3IOUS1024N.csv'))

    # sparse masks
    sparse_mask_path = os.path.join(sparse_path_root, 'Masks', sparsity)
    sparse_mask1 = get_mask(sparse_mask_path, cluster=1).t()
    sparse_mask2 = get_mask(sparse_mask_path, cluster=2).t()
    sparse_mask3 = get_mask(sparse_mask_path, cluster=3).t()
    sparse_dead  = [] #get_subactiv(sparse_mask_path, cluster=c)
    print(f"Sparse masks loaded — shapes: {sparse_mask1.shape}, {sparse_mask2.shape}, {sparse_mask3.shape}")

    # sparse model weights
    sparse_model_finallayerweights = torch.load(
        f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth',
        map_location=device
    )['state_dict']['mlp.3.weight']

    if method == 'CoFi':
        zs = torch.load(
            f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt',
            map_location=device
        )
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
            zs['final_mlp_hidden_z'].to(device)
        )


    # ── per-sample alignment ──────────────────────────────────────────────────
    pairwise_alignment  = {1: defaultdict(float), 2: defaultdict(float), 3: defaultdict(float), 'global':{}}
    holistic_alignment  = {1: defaultdict(float), 2: defaultdict(float), 3: defaultdict(float), 'global':{}}

    cluster_masks_sparse = {1: (sparse_mask1, sparse_highest_neurons1 if False else None),
                            2: (sparse_mask2, None),
                            3: (sparse_mask3, None)}
    cluster_masks_dense  = {1: (dense_mask1,  None),
                            2: (dense_mask2,  None),
                            3: (dense_mask3,  None)}
    avg=defaultdict(float)
    c1avg = []
    c1_expls=[]
    c2avg = []
    c2_expls=[]
    c3avg = []
    c3_expls=[]
    gavg = []
    gavg_expls = []
    da=masked(dense_activations)
    sa=masked(sparse_activations)
    '''for neuron in range(1024):
        if torch.sum(sparse_mask1[neuron]) >0 and torch.sum(dense_mask1[neuron]) > 0:
            c1avg.append(iou(sparse_mask1[neuron], dense_mask1[neuron]))
        
        if neuron in sparse_expls1 and neuron in dense_expls1:
            #print(sparse_expls1[neuron] & dense_expls1[neuron], sparse_expls1[neuron]  | dense_expls1[neuron])
            c1_expls.append(len(sparse_expls1[neuron] & dense_expls1[neuron]) / len(sparse_expls1[neuron]  | dense_expls1[neuron]))
            
        if torch.sum(sparse_mask2[neuron]) >0 and torch.sum(dense_mask2[neuron]) > 0:
            c2avg.append(iou(sparse_mask2[neuron], dense_mask2[neuron]))
        if neuron in sparse_expls2 and neuron in dense_expls2:
            c2_expls.append(len(sparse_expls2[neuron] & dense_expls2[neuron]) / len(sparse_expls2[neuron]  | dense_expls2[neuron]))
            
        if torch.sum(sparse_mask3[neuron]) >0 and torch.sum(dense_mask3[neuron]) > 0:
            c3avg.append(iou(sparse_mask3[neuron], dense_mask3[neuron]))
        if neuron in sparse_expls3 and neuron in dense_expls3:
            c3_expls.append(len(sparse_expls3[neuron] & dense_expls3[neuron]) / len(sparse_expls3[neuron]  | dense_expls3[neuron]))
        if torch.sum(sa[neuron]) >0 and torch.sum(da[neuron]) > 0:
            s_union_all = set()
            d_union_all = set()

            for cl in [1, 2, 3]:
                s_union_all |= set().union(*[sparse_expls1.get(neuron,set()), sparse_expls2.get(neuron,set()), sparse_expls3.get(neuron,set())])
                d_union_all |= set().union(*[dense_expls1.get(neuron,set()), dense_expls2.get(neuron,set()), dense_expls3.get(neuron,set())])
              

            gavg.append(iou(sa, da))
            gavg_expls.append(len(s_union_all & d_union_all) / len(s_union_all  | d_union_all))
            
    print(f'c1: {sum(c1avg)/len(c1avg)}\t expls: {sum(c1_expls)/len(c1_expls)}\nc2: {sum(c2avg)/len(c2avg)} expls: {sum(c2_expls)/len(c2_expls)}\nc3:{sum(c3avg)/len(c3avg)} expls: {sum(c3_expls)/len(c3_expls)}\nglboa: {sum(gavg)/len(gavg)} expls: {sum(gavg_expls)/len(gavg_expls)}')
    continue'''
    for c2w_sample in range(2):
        # top neurons per cluster — sparse
        sparse_top = {
            1: find_highest_activating_neurons_at_cluster(
                sparse_mask1.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
            2: find_highest_activating_neurons_at_cluster(
                sparse_mask2.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
            3: find_highest_activating_neurons_at_cluster(
                sparse_mask3.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
            'global': find_highest_activating_neurons(sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        }
        
        #import random

        complement = list(set(range(1024)) - set(sparse_neurons) - set(dense_neurons))
        
        sparse_top = {
            1: random.sample(complement, len(sparse_neurons)),
            2: random.sample(complement, len(sparse_neurons)),
            3: random.sample(complement, len(sparse_neurons)),
            'global': random.sample(complement, len(sparse_neurons))
        }
        

        # top neurons per cluster — dense
        dense_top = {
            1: find_highest_activating_neurons_at_cluster(
                dense_mask1.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
            2: find_highest_activating_neurons_at_cluster(
                dense_mask2.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
            3: find_highest_activating_neurons_at_cluster(
                dense_mask3.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
            'global': find_highest_activating_neurons(dense_activations[c2w_sample], dense_model_finallayerweights, dense_dead)
            
        }
       
     
       

        sparse_masks = {1: sparse_mask1, 2: sparse_mask2, 3: sparse_mask3, 'global': masked(sparse_activations.t())}
        dense_masks  = {1: dense_mask1,  2: dense_mask2,  3: dense_mask3,  'global': masked(dense_activations.t())}
        for cl in [1, 2, 3, 'global']:
            if cl == 'global':
                ioufunc =activationsiou
            else:
                ioufunc = iou
            
            avg[cl] += len(set(sparse_top[cl]) & set(dense_top[cl]))/len(dense_top[cl])
            # 1. pairwise alignment
            #print(set(sparse_top[cl]) & set(dense_top[cl]))
            pairwise_scores = [
                ioufunc(sparse_masks[cl][sparse_n], dense_masks[cl][dense_n]).item()
                for sparse_n, dense_n in zip(sparse_top[cl][:10], dense_top[cl][:10])
            ]
            for sparse_neuron in sparse_top[cl][:10]:
                print(f"Cluster {cl}, Num samples that activate neuron {sparse_neuron}: {torch.where(sparse_masks[cl][sparse_neuron] ==1,1,0).sum()}")
            pairwise_alignment[cl][c2w_sample] = sum(pairwise_scores) / len(pairwise_scores) \
                if pairwise_scores else 0.0
            
            sparse_union = None
            for n in sparse_top[cl][:10]:
                sparse_union = sparse_masks[cl][n] if sparse_union is None \
                    else (sparse_union | sparse_masks[cl][n])
            print(torch.where(sparse_union==1,1,0).sum())

            dense_union = None
            for n in dense_top[cl][:10]:
                dense_union = dense_masks[cl][n] if dense_union is None \
                    else (dense_union | dense_masks[cl][n])
            #print(sparse_union.sum())
            holistic_alignment[cl][c2w_sample] = iou(sparse_union, dense_union).item() \
                if (sparse_union is not None and dense_union is not None) else 0.0

    print({cl: a/600 for cl, a in avg.items()})
    # ── summary for this sparsity level ──────────────────────────────────────
    print(f"\n[{sparsity}]")
    for cl in [1, 2, 3, 'global']:
        avg_pair     = sum(pairwise_alignment[cl].values())  / len(pairwise_alignment[cl])
        avg_holistic = sum(holistic_alignment[cl].values())  / len(holistic_alignment[cl])
        print(f"  Cluster {cl} — Pairwise: {avg_pair:.4f}  Holistic: {avg_holistic:.4f}")

    start += 1

BERT lottery_ticket CLUSTER 1
Dense masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])

── 25.0%Pruned (Pruning iter 1) ──
Sparse masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])


/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)
/tmp/ipykernel_718/2919101027.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


Cluster 1, Num samples that activate neuron 270: 2410
Cluster 1, Num samples that activate neuron 668: 2241
Cluster 1, Num samples that activate neuron 1020: 2552
Cluster 1, Num samples that activate neuron 555: 2235
Cluster 1, Num samples that activate neuron 16: 2655
Cluster 1, Num samples that activate neuron 309: 2679
Cluster 1, Num samples that activate neuron 992: 2432
Cluster 1, Num samples that activate neuron 1: 2555
Cluster 1, Num samples that activate neuron 179: 2259
Cluster 1, Num samples that activate neuron 789: 2418
tensor(9321)
Cluster 2, Num samples that activate neuron 374: 1889
Cluster 2, Num samples that activate neuron 625: 2205
Cluster 2, Num samples that activate neuron 363: 1396
Cluster 2, Num samples that activate neuron 696: 1532
Cluster 2, Num samples that activate neuron 136: 1802
Cluster 2, Num samples that activate neuron 477: 2280
Cluster 2, Num samples that activate neuron 68: 1790
Cluster 2, Num samples that activate neuron 126: 2014
Cluster 2, Num sam

/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)
/tmp/ipykernel_718/2919101027.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


Cluster 1, Num samples that activate neuron 770: 2450
Cluster 1, Num samples that activate neuron 179: 2085
Cluster 1, Num samples that activate neuron 532: 2433
Cluster 1, Num samples that activate neuron 683: 2469
Cluster 1, Num samples that activate neuron 976: 2267
Cluster 1, Num samples that activate neuron 196: 2153
Cluster 1, Num samples that activate neuron 954: 2418
Cluster 1, Num samples that activate neuron 719: 2385
Cluster 1, Num samples that activate neuron 595: 2417
Cluster 1, Num samples that activate neuron 895: 2336
tensor(9156)
Cluster 2, Num samples that activate neuron 945: 1499
Cluster 2, Num samples that activate neuron 141: 1803
Cluster 2, Num samples that activate neuron 74: 2254
Cluster 2, Num samples that activate neuron 344: 2042
Cluster 2, Num samples that activate neuron 539: 1666
Cluster 2, Num samples that activate neuron 754: 1580
Cluster 2, Num samples that activate neuron 655: 1839
Cluster 2, Num samples that activate neuron 197: 1739
Cluster 2, Num s

/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)
/tmp/ipykernel_718/2919101027.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


Cluster 1, Num samples that activate neuron 959: 2157
Cluster 1, Num samples that activate neuron 878: 2794
Cluster 1, Num samples that activate neuron 440: 2488
Cluster 1, Num samples that activate neuron 885: 2412
Cluster 1, Num samples that activate neuron 495: 2422
Cluster 1, Num samples that activate neuron 867: 2435
Cluster 1, Num samples that activate neuron 620: 2182
Cluster 1, Num samples that activate neuron 106: 2335
Cluster 1, Num samples that activate neuron 33: 2354
Cluster 1, Num samples that activate neuron 197: 2629
tensor(9262)
Cluster 2, Num samples that activate neuron 1014: 2270
Cluster 2, Num samples that activate neuron 238: 2111
Cluster 2, Num samples that activate neuron 893: 1870
Cluster 2, Num samples that activate neuron 398: 2098
Cluster 2, Num samples that activate neuron 198: 2273
Cluster 2, Num samples that activate neuron 64: 2204
Cluster 2, Num samples that activate neuron 801: 2032
Cluster 2, Num samples that activate neuron 312: 1326
Cluster 2, Num s

/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)
/tmp/ipykernel_718/2919101027.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


Cluster 1, Num samples that activate neuron 91: 2527
Cluster 1, Num samples that activate neuron 508: 2510
Cluster 1, Num samples that activate neuron 120: 2589
Cluster 1, Num samples that activate neuron 831: 2567
Cluster 1, Num samples that activate neuron 131: 2409
Cluster 1, Num samples that activate neuron 112: 2192
Cluster 1, Num samples that activate neuron 705: 2421
Cluster 1, Num samples that activate neuron 1013: 2353
Cluster 1, Num samples that activate neuron 668: 2405
Cluster 1, Num samples that activate neuron 896: 2565
tensor(9355)
Cluster 2, Num samples that activate neuron 117: 1555
Cluster 2, Num samples that activate neuron 846: 1486
Cluster 2, Num samples that activate neuron 201: 1410
Cluster 2, Num samples that activate neuron 229: 2047
Cluster 2, Num samples that activate neuron 825: 1832
Cluster 2, Num samples that activate neuron 803: 1783
Cluster 2, Num samples that activate neuron 772: 2136
Cluster 2, Num samples that activate neuron 336: 1637
Cluster 2, Num 

/tmp/ipykernel_718/2919101027.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  activations = torch.tensor(samples_activations).abs()  # (1024,)
/tmp/ipykernel_718/2919101027.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask        = torch.tensor(cluster_mask)                # (1024,)
/tmp/ipykernel_718/2919101027.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights



[25.0%Pruned]
  Cluster 1 — Pairwise: 0.2047   Holistic: 1.0000
  Cluster 2 — Pairwise: 0.2227   Holistic: 1.0000
  Cluster 3 — Pairwise: 0.2662   Holistic: 0.7935

── 43.75%Pruned (Pruning iter 2) ──
Sparse masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])
/tmp/ipykernel_718/3365895126.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights

[43.75%Pruned]
  Cluster 1 — Pairwise: 0.2047   Holistic: 1.0000
  Cluster 2 — Pairwise: 0.2221   Holistic: 1.0000
  Cluster 3 — Pairwise: 0.2664   Holistic: 0.8094

── 57.812%Pruned (Pruning iter 3) ──
Sparse masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])
/tmp/ipykernel_718/3365895126.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights

[57.812%Pruned]
  Cluster 1 — Pairwise: 0.1994   Holistic: 1.0000
  Cluster 2 — Pairwise: 0.2142   Holistic: 1.0000
  Cluster 3 — Pairwise: 0.2524   Holistic: 0.8026

── 68.359%Pruned (Pruning iter 4) ──
Sparse masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])
/tmp/ipykernel_718/3365895126.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights

[68.359%Pruned]
  Cluster 1 — Pairwise: 0.1930   Holistic: 1.0000
  Cluster 2 — Pairwise: 0.2066   Holistic: 1.0000
  Cluster 3 — Pairwise: 0.2388   Holistic: 0.8010

── 76.27%Pruned (Pruning iter 5) ──
Sparse masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])
/tmp/ipykernel_718/3365895126.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights

[76.27%Pruned]
  Cluster 1 — Pairwise: 0.1838   Holistic: 1.0000
  Cluster 2 — Pairwise: 0.1951   Holistic: 1.0000
  Cluster 3 — Pairwise: 0.2159   Holistic: 0.7868


In [270]:
'''All c->w samples
Get highest activating neurons
Open mask for respective nerun
Calc alignment
Open expls for respective neuron
Clac concept dif %
At end: avg alignment and average concept dif %
If dif is high (close to 1) -> dif concepts entirely 
Alignment high: neurons behave the same so theyre learning dif ways to represent the same set → but those details are the wrong way
Alignment low: neurons explain vastly dif concepts and fire for very dif samples so the behavior of the neuron changes fully → wrong behavior
If dif is low (less than 50) -> dif combos 
High align: same behsvior even when misclassified
Low: wrong combos!'''

model='BERT'
method='CoFi'
c=1

print(f"{model} {method} CLUSTER {c}")
path_to_experiment=os.path.join("/workspace/CCE_NLI",model, 'exp', method, 'Run0.25_5')
#dense_cw = pd.read_csv(os.path.join(path_to_experiment,  f'Prediction_CW_0_Pruning_Iter.csv')).set_index('Unnamed: 0').T
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
dense_expls1,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{1}IOUS1024N.csv'))
dense_expls2,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{2}IOUS1024N.csv'))
dense_expls3,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{3}IOUS1024N.csv'))
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))
    
    
dense_path_root = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
dense_mask1 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=1)
dense_mask2 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=2)
dense_mask3 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=3)
start=1
dense_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
dense_dead=get_subactiv(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=c)
if '0.0%Pruned' in os.listdir(f"{sparse_path_root}/Expls"):
    start=1
    
sparse_path_root=f"{path_to_experiment}"
for i, sparsity in enumerate(sorted(os.listdir(f"{path_to_experiment}/Expls"))):

    print(sparse_path_root, sparsity)
    #if sparsity != '25.0%Pruned': continue
    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity: continue
    #sparse_cw = pd.read_csv(os.path.join(path_to_experiment, f'Prediction_CW_{start}_Pruning_Iter.csv')).set_index('Unnamed: 0').T
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    sparse_expls_1, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{1}IOUS1024N.csv'))
    sparse_expls_2, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{2}IOUS1024N.csv'))
    sparse_expls_3, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{3}IOUS1024N.csv'))
    
  
    
    sparse_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
    
    if method=='CoFi':
        zs=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt', map_location=device)
     
        print(zs['final_mlp_hidden_z'].device, sparse_model_finallayerweights.device)
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(zs['final_mlp_hidden_z'].to(device))
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))
        
    #c_2_w = all_correct_to_wrong(dense_cw, sparse_cw)
    
    per_neuron_average_alignment = {'1':defaultdict(list), '2':defaultdict(list), '3':defaultdict(list)}
    avg_cluster_holistic_alignment = 0
    
    alignment_between_neurons= []
    average_concept_diff = defaultdict(int)
    sparse_mask = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=c)
    print(f"LOADED FROM FILE SHPAE {sparse_mask.shape}")
    sparse_dead = get_subactiv(os.path.join(dense_path_root, 'Masks', sparsity), cluster=c)
    start += 1
    #print(sparsity, len(c_2_w))
    died_neurons=0
    revived_neurons=0
    dense_highest_neuronsset=set()
    sparse_mask_1 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=1)
    sparse_mask_2 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=2)
    sparse_mask_3 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=3)
       
        
    dense_highest_neurons1 = find_highest_activating_neurons_at_cluster(dense_mask_1[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons1 =set(dense_highest_neurons1)
    dense_highest_neurons2 = find_highest_activating_neurons_at_cluster(dense_mask_2[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons2=set(dense_highest_neurons2)
    dense_highest_neurons3 = find_highest_activating_neurons_at_cluster(dense_mask_3[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons3=set(dense_highest_neurons3)
        
    for c2w_sample in range(300): #[:100]:
 

   
        difference_in_concepts=0
        ignore = 0
       
        sparse_highest_neurons1 = find_highest_activating_neurons_at_cluster(sparse_mask_1[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons1=set(sparse_highest_neurons1)
        sparse_highest_neurons2= find_highest_activating_neurons_at_cluster(sparse_mask_2[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons2=set(sparse_highest_neurons2)
        sparse_highest_neurons3= find_highest_activating_neurons_at_cluster(sparse_mask_3[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons3 = set(sparse_highest_neurons3)
        holisitc_sparse = sparse_highest_neurons1.union(sparse_highest_neurons2).union(sparse_highest_neurons3)

        
        hol_sparse_highest_neurons = find_highest_activating_neurons( sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        hol_dense_highest_neurons = find_highest_activating_neurons(dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons3, dense_highest_neurons3):  
            per_neuron_alignment=iou(sparse_mask_3[sparse_highest_neuron], dense_mask3[dense_highest_neuron])
            per_neuron_average_alignment['3'][c2w_sample].append(per_neuron_alignment.item())
            
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons2, dense_highest_neurons2):  
            per_neuron_alignment=iou(sparse_mask_2[sparse_highest_neuron], dense_mask2[dense_highest_neuron])
            per_neuron_average_alignment['2'][c2w_sample].append(per_neuron_alignment.item())
            
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons1, dense_highest_neurons1):  
            per_neuron_alignment=iou(sparse_mask_1[sparse_highest_neuron], dense_mask1[dense_highest_neuron])
            per_neuron_average_alignment['1'][c2w_sample].append(per_neuron_alignment.item())
            
        holisitc_sparse=list(holisitc_sparse)
        holisitc_dense=list(holisitc_dense)
        
        sparse_union_mask = sparse_activations.t()[holisitc_sparse].any(dim=0)
        dense_union_mask = dense_activations.t()[holisitc_dense].any(dim=0)
        avg_cluster_holistic_alignment  += activationsiou(sparse_union_mask, dense_union_mask)
            
           
            
        for cluster in per_neuron_average_alignment.keys():
            per_neuron_average_alignment[cluster][c2w_sample] = sum(per_neuron_average_alignment[cluster][c2w_sample])/len(per_neuron_average_alignment[cluster][c2w_sample])
    for cluster in per_neuron_average_alignment.keys():
        per_neuron_average_alignment[cluster]= sum(per_neuron_average_alignment[cluster].values())/len(per_neuron_average_alignment[cluster])
        
    print(f"per neuron Average alignment: ", per_neuron_average_alignment)
    print(f"Holistic average alignment: ", avg_cluster_holistic_alignment / 300)
        
   #for sample 8204 720 is the highst contributiig dense neuron and 718 is the highest contributiig sparse neruon (25%)  so 
        #see where these samples differ in the firing. (collect all the sentences into a txt file (2 sep ) and compare them see if any patterns)

BERT lottery_ticket CLUSTER 1
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5 0.0%Pruned
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5 25.0%Pruned
LOADED FROM FILE SHPAE torch.Size([10000, 1024])


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights
/tmp/ipykernel_350/211658876.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


KeyboardInterrupt: 

In [ ]:
no dif between preservation: neurons behave dif at all sparsities regardless of how pred changed
bert lth c->c
25.0%Pruned 8290
Average concept difference :  0.6654999999999999
Average alignment:  0.23897429130971432
    
43.75%Pruned 7811
Average concept difference :  0.6144999999999998
Average alignment:  0.17982538217678667

57.812%Pruned 6156
Average concept difference :  0.6798333333333332
Average alignment:  0.17942780851386486
    
68.359%Pruned 3780
Average concept difference :  0.7386666666666666
Average alignment:  0.1512690109340474
    
76.27%Pruned 2600
Average concept difference :  0.6961666666666667
Average alignment:  0.1716563373338431

    
bert lth c->w
25.0%Pruned 136
Average concept difference :  0.6161666666666665
Average alignment:  0.23048584141070022

43.75%Pruned 615
Average concept difference :  0.6861666666666666
Average alignment:  0.21354142345953733
    
57.812%Pruned 2270
Average concept difference :  0.7136666666666664
Average alignment:  0.1814640353480354
    
68.359%Pruned 4646
Average concept difference :  0.7028333333333333
Average alignment:  0.1619819349143654
    
76.27%Pruned 5826
Average concept difference :  0.6745
Average alignment:  0.16334903969895095


In [16]:
bert cofi c->w
0.25267370011269075%
Average concept difference :  54.25
Average alignment:  0.39514948163181546

0.4443280775114645%
Average concept difference :  59.28333333333332
Average alignment:  0.3604531835205853

0.5826748728764346%
Average concept difference :  46.649999999999997
Average alignment:  0.402138639902696

0.6841850626856109%Pruned 583
Average concept difference :  52.63333333333334
Average alignment:  0.3442400005261879

0.7789752992375562%Pruned 655
Average concept difference :  54.58333333333332
Average alignment:  0.29560177197214216
    
    
bert cofi c->c (at earlier iters concrpt dif is 10% lower & alignment is 6% higher. at 58 it shifts to more dif behavior)
so initially: sample i wrongly classif because it placed emphasus on a suboptimal neuron (ie pruning made the neuron less optimal for tha sample or removed kpwledge of the samole)
later: theres no dif numerically, neuron behavior changes a lot regrdless
    
0.25267370011269075%Pruned 7957
Average concept difference :  45.666666666666667
Average alignment:  0.452997426581569
   
0.4443280775114645%Pruned 7986
Average concept difference :  47.98333333333333
Average alignment:  0.42837691662833094
    
0.5826748728764346%Pruned 7869
Average concept difference :  53.01666666666667
Average alignment:  0.34265444518066945
    
0.6841850626856109%Pruned 7725
Average concept difference :  52.36666666666666
Average alignment:  0.3495491479942575
    
0.7789752992375562%Pruned 7653
Average concept difference :  51.8
Average alignment:  0.2954905626235995

tensor([ 0,  3,  6,  9, 12])

In [98]:
abs_map[2]

['cucumbers',
 'broccoli',
 'tasty',
 'pepper',
 'onion',
 'barney',
 'patrick',
 'mcdonalds',
 'avocados',
 'seafood',
 'strawberry',
 'lemons',
 'peppermints',
 'chili',
 'kfc',
 'banana',
 'ingredients',
 'nike',
 'joshua',
 'fruits',
 'nutritious',
 'tomatoes',
 'vegetable',
 'lemonade',
 'oranges',
 'fruit',
 'donald',
 'apple',
 'vegetables',
 'stanley',
 'coca',
 'soda',
 'cola',
 'blueberry',
 'fresh',
 'asparagus',
 'berries',
 'carrots',
 'cherry',
 'tomato',
 'hickock',
 'mcdonald',
 'foods',
 'juice',
 'carrot',
 'heineken',
 'ginger',
 'hershey',
 'delicious',
 'coconuts',
 'beverage',
 'lemon',
 'jeffs',
 'coconut',
 'apples',
 'john',
 'bananas',
 'chris',
 'maple',
 'grapes',
 'mary',
 'lime',
 'food',
 'pineapples',
 'freshly']

holisitc alignment

In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict, Counter
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import json
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
from itertools import combinations

def find_highest_activating_neurons(samples_activations, model_finallayerweights, d):
    flweights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    flweights = flweights.reshape((3, flweights.shape[0]//3))
    
    contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights
    
    total = contribution.sum(dim=1).argmax()  # best class row
    
    class_contributions = contribution[total]  # [1024]
    
    sorted_neurons = class_contributions.argsort(descending=True)  # indices sorted by contribution
    active_sorted_neurons = sorted_neurons[class_contributions[sorted_neurons] > 0]  # filter only active
    
    return active_sorted_neurons.tolist()
    

def find_highest_activating_neurons_at_cluster(cluster_mask, samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
   
    
    fl_cluster_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    
    fl_cluster_weights=fl_cluster_weights.reshape((3,fl_cluster_weights.shape[0]//3 ))
  
    contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    class_contributions = contribution[total]  # [1024]
    
    sorted_neurons = class_contributions.argsort(descending=True)  # indices sorted by contribution
    active_sorted_neurons = sorted_neurons[class_contributions[sorted_neurons] > 0]  # filter only active
    
    return active_sorted_neurons.tolist()

 

model='BERT'
method='lottery_ticket'
c=3

print(f"{model} {method} CLUSTER {c}")
path_to_experiment=os.path.join("/workspace/CCE_NLI",model, 'exp', method, 'Run0.25_5')
#dense_cw = pd.read_csv(os.path.join(path_to_experiment,  f'Prediction_CW_0_Pruning_Iter.csv')).set_index('Unnamed: 0').T
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
dense_expls_1,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster1IOUS1024N.csv'))
dense_expls_2,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster2IOUS1024N.csv'))
dense_expls_3,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster3IOUS1024N.csv'))
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))
    
    
dense_path_root = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root = f"/workspace/CCE_NLI/{model}/exp/{method}/Run0.25_5"
dense_mask_1 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=1)
dense_mask_2 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=2)
dense_mask_3 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=3)

start=1
dense_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
dense_dead=get_subactiv(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=c)
if '0.0%Pruned' in os.listdir(f"{sparse_path_root}/Expls"):
    start=1
    
for i, sparsity in enumerate(sorted(os.listdir(f"{path_to_experiment}/Expls"))):
   
    
    #if sparsity != '25.0%Pruned': continue
    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity: continue
    #sparse_cw = pd.read_csv(os.path.join(path_to_experiment, f'Prediction_CW_{start}_Pruning_Iter.csv')).set_index('Unnamed: 0').T
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    sparse_expls_1, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{1}IOUS1024N.csv'))
    sparse_expls_2, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{2}IOUS1024N.csv'))
    sparse_expls_3, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{3}IOUS1024N.csv'))
    
    
    sparse_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
    
    if method=='CoFi' and start>0:
        zs=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt', map_location=device)
        print(zs['final_mlp_hidden_z'].device, sparse_model_finallayerweights.device)
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(zs['final_mlp_hidden_z'].to(device))
    
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))
        
    #c_2_w = all_correct_to_wrong(dense_cw, sparse_cw)
    
    average_alignment = defaultdict(list)
    alignment_between_neurons= []
    average_concept_diff = defaultdict(int)
    sparse_mask_1 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=1)
    sparse_mask_2 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=2)
    sparse_mask_3 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=3)
  
    #sparse_dead = get_subactiv(os.path.join(dense_path_root, 'Masks', sparsity), cluster=c)
    start += 1
    #print(sparsity, len(c_2_w))
    died_neurons=0
    revived_neurons=0
    dense_highest_neuronsset=set()
    avg_cluster_holistic_alignment= 0
    concept_hol_dif=0
    for c2w_sample in range(600): #[:100]:
        sparse_highest_neurons1 = find_highest_activating_neurons_at_cluster(sparse_mask_1[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons2= find_highest_activating_neurons_at_cluster(sparse_mask_2[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons3= find_highest_activating_neurons_at_cluster(sparse_mask_3[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
      
        dense_highest_neurons1 = find_highest_activating_neurons_at_cluster(dense_mask_1[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        dense_highest_neurons2 = find_highest_activating_neurons_at_cluster(dense_mask_2[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        dense_highest_neurons3 = find_highest_activating_neurons_at_cluster(dense_mask_3[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        
        difference_in_concepts=0
      
        #sparrse_expls_hol_covered =set().union(*[sparse_expls_1[n] for n in sparse_highest_neurons1])
        #sparrse_expls_hol_covered =set().union(*[sparse_expls_2[n] for n in sparse_highest_neurons2])
        sparrse_expls_hol_covered =set().union(*[sparse_expls_3[n] for n in sparse_highest_neurons3])
       
        #dense_expls_hol_covered = set().union(*[dense_expls_1[n] for n in dense_highest_neurons1])
        #dense_expls_hol_covered = set().union(*[dense_expls_2[n] for n in dense_highest_neurons2])
        dense_expls_hol_covered = set().union(*[dense_expls_3[n] for n in dense_highest_neurons3])
        
        
        concept_hol_dif += len(sparrse_expls_hol_covered & dense_expls_hol_covered)/len(dense_expls_hol_covered)
        #print(sorted(sparrse_expls_hol_covered & dense_expls_hol_covered))
        
        sparse_union_mask = sparse_mask_3.t()[sparse_highest_neurons3].any(dim=0)
        dense_union_mask = dense_mask_3.t()[dense_highest_neurons3].any(dim=0)
        
        avg_cluster_holistic_alignment  += activationsiou(sparse_union_mask,dense_union_mask )
        print("Cluster 3: ", iou(sparse_union_mask,dense_union_mask ))
        
    print("iou between sparse concept abstraction cverage and dense", sparsity, concept_hol_dif/600, avg_cluster_holistic_alignment/600)    

In [ ]:
llama lth
    alginment of cluste 1 for equiv neruons
    1
    1
    1
    1
    1
    1

    cluster 2:
    1
    1
    1
    1
    1

    how alignment are the cluster 3 masks for equiv neurons:
    0.7818
    0.7730
    0.7823
    0.7715
    0.7754
    
    
    
alignment of behavior of all neurons that fire at the respective cluster
bert lth
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons:
    0.8135
    0.8013
    0.8054
    0.8027
    0.8013
    
bert wanda:
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons: 
    0.8928
    0.8160
    0.7956
    0.8351
    0.8373
    
bert cofi
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons:
    0.8485
    0.8431
    0.8312
    0.8169
    0.7996

In [ ]:
bert wanda percnet sparse in dense with firing neurons t c3 holistically 
0.7158253470690867
0.5158081245464058
0.3367145398698728
0.3319527369198622
0.3217954856318257

bert wanda percnet sparse in dense with firing neurons t c3 holistically 
0.780325488503813
0.6618895856757624
0.49561810158466124
0.4297488691264148
0.387621722854215

bert lth percnet sparse in dense with firing neurons all clusters holistically  idnic concepst
0.7157827434841774
0.7150289015989116
0.705904915232019
0.7144492558370446
0.7104443351851716

bert wanda percnet sparse in dense with firing neurons holistically  (indiv concepts across all clusters)
0.7646508236874388
0.6824849638716762
0.5811244901600291
0.5204990223352913
0.49750395413909293

bert wanda percent groupings preserverd across all cluster:
25.0%Pruned 0.47863799756537856
43.75%Pruned 0.3298689137169994
57.812%Pruned 0.17422260587626442
68.359%Pruned 0.1283690198004774
76.27%Pruned 0.11109980672192672

so holistocally its not learning the same combos either at same cluster or acros clusters
its learning the same concepts but those are also in the untrained so its not indicative that the same concepst translate to same meaning

holsitcially cofi,lth is relearning the same absrtactions but it has that alignment even with pretrained or untrained (80-90%) 
    so its not indicative that the same concepst translate to same meaning

holsitically lth is also learning different groupings (35%avg fro lth with dense but 8%/ 12->14% w/ pretrained/untrained) 
holsitically cofi is also learning different groupings (38->33% fro cofi with dense but 8%/ 12->14% w/ pretrained/untrained) 
even wanda learns different grouppings which become more different with more prunign



basicaclly same indic concepts, diff groupings, same abstractions but same accuracy in lth/cofi
llama preserved along all firing neurons
    LTH groups
        0.30236508537915024
        0.2791821486572037
        0.2723068843244222
        0.27068421167371604
        0..


bert preserved along all firing neurons
    LTH indiv concepts 
        0.6979689263479909
        0.6975141678956701
        0.6900088414758995
        0.6972155524542846
        0.6933007260580186
    LTH groups
        0.3259912533643099
        0.3135241959204562
        0.31154873128645416
        0.30749358750641015
        0.31431311428325825
    LTH absracaiton
        0.9492857142857122
        0.9716666666666656
        0.9478571428571403
        0.9771428571428566
        0.9511904761904737
    
    COFI indiv concepts
        0.7400342995328111
        0.740564712147953
        0.7202402188955461
        0.7116983488704783
        0.6932507743365783
    COFI groiups
        0.3916439404098978
        0.391764689701342
        0.38044205211166937
        0.3555609300508366
        0.3396628598863317
    COFI abstacints
        0.9628571428571405
        0.9621428571428547
        0.9599999999999977
        0.9519047619047591
        0.9480952380952349
                
    WANDA indiv concets
        0.7589840355837185
        0.6803471127028979
        0.5770340951137035
        0.5061291564181638
        0.48619111751925287
    WANDA grouos
        0.45621053550492385
        0.3171332028348171
        0.17270628580981573
        0.1302600391771703
        0.11701319797904672
    WANDA abstracion
        0.9502380952380928
        0.9792857142857133
        0.9140873015872991
        0.848412698412695
        0.893531746031742
        
all preserves concepts
    - cofi does most, lth, then wanda (onlu low spasrsities)
none preserves groups
all preserves abstactions


all: preserves concepts and abstractions not goups  (Wanda only preserves concepts at low sparsities and higher sparsities become mroe different)
cofi: better job of preserving concepts and groups than lth and takes less space